In [1]:
!pip install PyPortfolioOpt
!pip install scikit-learn
!pip install stable-baselines3
!pip install gymnasium
!pip install pypfopt
!pip install tqdm
!pip install torch
!pip install torchvision

ERROR: Could not find a version that satisfies the requirement pypfopt (from versions: none)
ERROR: No matching distribution found for pypfopt


Reference: 

First Prompt(Anand): Help me configure gynasium environment - training AI-based portfolio managers, enabling them to learn optimal portfolio rebalancing strategies over time.

Last Prompt(Anand): Evaluate a trained Deep Reinforcement Learning (DRL) model on unseen stock market data to assess its performance. The evaluation should include:

Performance Metrics Calculation:
Sharpe Ratio (risk-adjusted return).
Max Drawdown (largest peak-to-trough decline).
Total Return (overall portfolio growth).
Comparison Against Benchmark:
Compare portfolio performance with a market index (e.g., S&P 500).
Analyze risk-adjusted returns.
Evaluation Process:
Load the model and test it on new stock data not seen during training.
Track portfolio value over time and compute daily returns.
Analyze win rate and volatility.
Output:
Print a summary of performance metrics.
Visualize results using charts (e.g., equity curve, drawdown plot, returns histogram).
Ensure robustness by handling potential errors (e.g., missing data, extreme price movements).

In [2]:
import numpy as np
import pandas as pd
import gymnasium as gym
from gymnasium import spaces
from stable_baselines3 import DDPG
from stable_baselines3.common.vec_env import DummyVecEnv
from sklearn.preprocessing import StandardScaler
import torch
from datetime import datetime
from stable_baselines3.common.noise import NormalActionNoise

# Stock Portfolio Data Iterator for Deep Reinforcement Learning

The StockPortfolioIterator class is a custom iterator designed for loading, processing, and iterating over historical stock market data for portfolio management using Deep Reinforcement Learning (DRL). It prepares stock price data by applying preprocessing techniques such as filtering based on trading history, computing technical indicators, and normalizing features.

This iterator enables efficient batch retrieval of stock price data over a lookback window, making it suitable for training reinforcement learning models that require sequential financial data. Key functionalities include:

Data Loading & Preprocessing: Reads stock market data from a CSV file, filters stocks based on minimum trading history and trading volume, and prepares a structured dataset.

Feature Engineering: Computes essential technical indicators like Moving Averages, RSI, MACD, Momentum, and Volatility.

Data Normalization: Scales numerical features for consistent input representation.

Efficient Data Storage & Access: Stores preprocessed data in a dictionary format for quick retrieval during training.

Iterative Batch Processing: Provides a sequential data stream with a defined lookback window for reinforcement learning agents.

This iterator simplifies data handling for stock portfolio optimization models and ensures that financial data is structured effectively for deep learning-based portfolio allocation strategies.

In [3]:
class StockPortfolioIterator:
    def __init__(self, file_path: str, lookback_window: int = 30, min_history: int = 100, max_stocks: int = 100, train_years: int = 20):

        print(f"Loading data from {file_path}...")
        self.data = pd.read_csv(file_path)
        self.data['date'] = pd.to_datetime(self.data['date'])

        # Only use recent data (last few years)
        cutoff_date = self.data['date'].max() - pd.DateOffset(years=train_years)
        self.data = self.data[self.data['date'] >= cutoff_date]

        self.data = self.data.sort_values(['date', 'tic'])

        # Filter stocks with enough history
        stock_counts = self.data.groupby('tic').size()
        valid_stocks = stock_counts[stock_counts >= min_history].index

        # Select stocks with highest trading volume (more likely to be liquid)
        if len(valid_stocks) > max_stocks:
            avg_volumes = self.data[self.data['tic'].isin(valid_stocks)].groupby('tic')['volume'].mean()
            valid_stocks = avg_volumes.nlargest(max_stocks).index.tolist()

        self.data = self.data[self.data['tic'].isin(valid_stocks)]
        self.unique_dates = sorted(self.data['date'].unique())
        self.tickers = sorted(self.data['tic'].unique())
        self.num_stocks = len(self.tickers)
        self.lookback = lookback_window
        self.num_features = 7

        # Create ticker lookup for faster access
        self.ticker_indices = {ticker: i for i, ticker in enumerate(self.tickers)}

        self._add_technical_indicators()
        self._scale_features()

        # Store data in a more efficient format
        self.prepare_training_data()

        print(f"Loaded {self.num_stocks} stocks with {len(self.unique_dates)} trading days")
        print(f"Date range: {self.unique_dates[0]} to {self.unique_dates[-1]}")

    def _add_technical_indicators(self):

        print("Adding technical indicators...")
        indicators = ['Returns', 'Volume_MA', 'Price_MA', 'RSI', 'MACD', 'Volatility', 'Momentum']

        # Process one ticker at a time to avoid memory issues
        for tic in self.tickers:
            mask = self.data['tic'] == tic
            stock_data = self.data.loc[mask].copy()

            # Calculate technical indicators
            stock_data['Returns'] = stock_data['close'].pct_change()
            stock_data['Price_MA'] = stock_data['close'].rolling(window=20, min_periods=1).mean()
            stock_data['Momentum'] = stock_data['close'].pct_change(periods=10)
            stock_data['Volume_MA'] = stock_data['volume'].rolling(window=10, min_periods=1).mean()

            # RSI Calculation
            delta = stock_data['close'].diff()
            gain = (delta.clip(lower=0)).rolling(window=14, min_periods=1).mean()
            loss = (-delta.clip(upper=0)).rolling(window=14, min_periods=1).mean()
            rs = gain / (loss + 1e-8)
            stock_data['RSI'] = 100 - (100 / (1 + rs))

            # MACD Calculation
            exp1 = stock_data['close'].ewm(span=12, adjust=False).mean()
            exp2 = stock_data['close'].ewm(span=26, adjust=False).mean()
            stock_data['MACD'] = exp1 - exp2

            # Volatility
            stock_data['Volatility'] = stock_data['Returns'].rolling(window=30, min_periods=1).std()

            # Update the main dataframe
            self.data.loc[mask, indicators] = stock_data[indicators]

        # Fill missing values
        self.data = self.data.fillna(0)
        print("Technical indicators added successfully")

    def _scale_features(self):
        print("Scaling features...")
        feature_columns = ['Returns', 'Volume_MA', 'Price_MA', 'RSI', 'MACD', 'Volatility', 'Momentum']

        for tic in self.tickers:
            mask = self.data['tic'] == tic
            scaler = StandardScaler()
            self.data.loc[mask, feature_columns] = scaler.fit_transform(self.data.loc[mask, feature_columns])

    def prepare_training_data(self):
        print("Preparing training data...")
        # Pre-allocate arrays for features and prices
        self.all_features = {}
        self.all_prices = {}
        feature_columns = ['Returns', 'Volume_MA', 'Price_MA', 'RSI', 'MACD', 'Volatility', 'Momentum']

        # Process data by date for efficient access during training
        for date_idx, date in enumerate(self.unique_dates):
            date_mask = self.data['date'] == date
            date_data = self.data[date_mask]

            features = np.zeros((self.num_stocks, self.num_features))
            prices = np.zeros(self.num_stocks)

            for _, row in date_data.iterrows():
                ticker_idx = self.ticker_indices.get(row['tic'])
                if ticker_idx is not None:
                    features[ticker_idx] = row[feature_columns].values
                    prices[ticker_idx] = row['close']

            self.all_features[date] = features
            self.all_prices[date] = prices

    def __iter__(self):
        """Initialize iterator"""
        self.current_idx = self.lookback
        return self

    def __next__(self):
        """Get next batch of data"""
        if self.current_idx >= len(self.unique_dates):
            raise StopIteration

        current_date = self.unique_dates[self.current_idx]
        window_dates = self.unique_dates[self.current_idx - self.lookback:self.current_idx]

        # Get lookback window features
        features = np.zeros((self.num_stocks, self.lookback, self.num_features), dtype=np.float32)

        for i, date in enumerate(window_dates):
            if date in self.all_features:
                features[:, i, :] = self.all_features[date]

        prices = self.all_prices[current_date]

        self.current_idx += 1
        return {
            'features': features,
            'prices': prices,
            'date': current_date,
            'tickers': self.tickers
        }

    def reset(self):
        """Reset the iterator"""
        self.current_idx = self.lookback
        return self

# Portfolio Recommender Using Deep Reinforcement Learning

Description:
The PortfolioRecommender class is an AI-powered financial tool that suggests optimal stock allocations using a trained Deep Deterministic Policy Gradient (DDPG) model. It processes the latest stock market data, computes relevant features, and generates investment recommendations based on reinforcement learning strategies.

This class is particularly useful for beginner investors looking to allocate a predefined investment amount into a diversified stock portfolio while keeping a portion in cash. The recommendations are generated by analyzing recent stock performance and computing optimal portfolio weights.

Key Features:
DDPG Model Integration: Loads a pre-trained Deep Reinforcement Learning (DRL) model for portfolio optimization.
Stock Selection: Filters the top liquid stocks based on trading volume to ensure reliable investments.
Feature Engineering: Computes key technical indicators, including returns, volatility, and momentum to ensure model consistency.
Smart Allocation Strategy:
Predicts portfolio weights for stocks and cash allocation.
Normalizes investment allocations to sum up to 100% of the input capital.
Ensures risk-adjusted diversification by using historical trends.
How It Works:
Loads Market Data → Reads the latest stock data and selects the top stocks based on liquidity.
Prepares Features → Extracts the most relevant technical indicators to match training data.
Predicts Portfolio Weights → Uses the DDPG model to determine the optimal stock allocation.
Generates Investment Plan → Converts model output into real-world investment recommendations:
Percentage allocation per stock
Amount invested per stock (in CAD)
Number of shares to buy
Remaining cash reserve
Ideal Use Case:
This tool is designed for investors with no prior trading experience who want AI-driven insights into portfolio allocation. It ensures an intelligent and data-driven approach to investing in stocks while minimizing risk and maximizing returns over time.

In [4]:
class PortfolioRecommender:
    def __init__(self, model_path, data_path, max_stocks=100, lookback=30):
        """Initialize the portfolio recommender with the trained DDPG model and latest stock data."""
        self.model = DDPG.load(model_path)  # Load DDPG model
        self.max_stocks = max_stocks
        self.lookback = lookback

        # Load and preprocess stock data
        data = pd.read_csv(data_path)
        data['date'] = pd.to_datetime(data['date'])

        # Get the most recent trading date
        self.latest_date = data['date'].max()

        # Select top stocks by trading volume (same filtering as in training)
        recent_data = data[data['date'] >= (self.latest_date - pd.DateOffset(days=30))]
        avg_volumes = recent_data.groupby('tic')['volume'].mean()
        top_tickers = avg_volumes.nlargest(max_stocks).index.tolist()

        # Get the ordered list of tickers (must match training data order)
        self.tickers = sorted(top_tickers)

        # Store latest closing prices for selected stocks
        latest_data = data[data['date'] == self.latest_date]
        self.latest_prices = {row['tic']: row['close'] for _, row in latest_data.iterrows()
                              if row['tic'] in self.tickers}

        # Compute technical indicators (ensures feature consistency)
        self._prepare_features(data)

    def _prepare_features(self, data):
        """Prepares feature matrix for the latest available stock data."""
        filtered_data = data[data['tic'].isin(self.tickers)]

        # Get the most recent dates for feature calculation
        recent_dates = sorted(filtered_data['date'].unique())[-self.lookback:]
        recent_data = filtered_data[filtered_data['date'].isin(recent_dates)]

        # Initialize the feature array
        features = np.zeros((self.max_stocks, self.lookback, 7), dtype=np.float32)

        for i, ticker in enumerate(self.tickers):
            if i >= self.max_stocks:
                break

            ticker_data = recent_data[recent_data['tic'] == ticker].sort_values('date')
            if len(ticker_data) == self.lookback:
                # Create feature array (must match training data features)
                ticker_features = np.zeros((self.lookback, 7))
                ticker_features[:, 0] = ticker_data['close'].pct_change().fillna(0).values  # Returns
                ticker_features[:, 1] = ticker_data['volume'].values / ticker_data['volume'].mean()  # Normalized Volume
                ticker_features[:, 2] = ticker_data['close'].values / ticker_data['close'].mean()  # Normalized Price
                ticker_features[:, 3] = 50  # Placeholder for RSI
                ticker_features[:, 4] = 0   # Placeholder for MACD
                ticker_features[:, 5] = ticker_data['close'].pct_change().rolling(5).std().fillna(0).values  # Volatility
                ticker_features[:, 6] = ticker_data['close'].pct_change(5).fillna(0).values  # Momentum

                features[i] = ticker_features

        self.recent_features = features

    def recommend_portfolio(self, amount_cad):
        """Generates stock allocation recommendations based on the trained DDPG model."""

        # Use the latest computed features for prediction
        action, _ = self.model.predict(self.recent_features, deterministic=True)

        # Normalize action values to sum to 1
        action = np.clip(action, 0, 1)
        if action.sum() > 0:
            action /= action.sum()

        # Allocate cash and stocks
        allocations = {}
        cash_allocation = action[0] * amount_cad  # First action is cash allocation

        for i, ticker in enumerate(self.tickers):
            if i >= len(self.tickers) or i+1 >= len(action):
                continue

            allocation = action[i+1] * amount_cad  # Skip first index (cash)
            if allocation > 0 and ticker in self.latest_prices:
                price = self.latest_prices[ticker]
                shares = allocation / price if price > 0 else 0

                allocations[ticker] = {
                    "allocation_percent": float(action[i+1] * 100),
                    "allocation_cad": float(allocation),
                    "price_per_share": float(price),
                    "shares": float(shares)
                }

        return {
            "cash_percent": float(action[0] * 100),
            "cash_amount": float(cash_allocation),
            "stock_allocations": allocations,
            "total_amount": float(amount_cad),
            "recommendation_date": str(self.latest_date)
        }



# Portfolio Trading Environment for Deep Reinforcement Learning

The PortfolioTradingEnv class defines a custom OpenAI Gym environment designed for training Deep Reinforcement Learning (DRL) agents (such as DDPG) to manage a stock portfolio. It simulates real-world stock trading conditions by allowing an agent to allocate capital across multiple stocks, adjusting its holdings dynamically while accounting for transaction costs.

This environment is crucial for training DRL-based portfolio optimization models, enabling them to learn investment strategies that maximize returns while minimizing risks.

Key Features:
State Representation:
Observations consist of technical indicators over a lookback window for multiple stocks.
Action Space:
A continuous action space where the model assigns portfolio weights across stocks and cash.
Portfolio Management Logic:
The agent decides how to allocate capital among stocks and cash.
The system executes trades based on the portfolio allocation.
Transaction costs are deducted based on trade volume.
Reward Mechanism:
The reward function is based on portfolio value appreciation after executing trades, penalizing transaction costs.
Episode Termination:
The episode ends when the dataset is exhausted, ensuring the model learns from historical market patterns.
How It Works:
Initialization (__init__):

Loads stock data using a data iterator.
Defines the action space (portfolio allocation) and observation space (historical stock features).
Sets up initial cash balance and stock holdings.
Reset (reset):

Resets cash and holdings to starting conditions.
Fetches the first set of stock data from the iterator.
Step (step):

Receives a portfolio allocation decision from the DRL agent.
Normalizes actions to ensure total allocation sums to 100%.
Computes new stock holdings based on allocation.
Deducts transaction costs for portfolio adjustments.
Computes the new portfolio value and reward.
Fetches the next trading day’s data.
Determines if the episode has ended.

Ideal Use Case:
This environment is designed for training AI-based portfolio managers, enabling them to learn optimal portfolio rebalancing strategies over time. It is particularly useful for automated trading systems and hedge funds aiming to improve portfolio performance using AI-driven decision-making.

In [5]:
class PortfolioTradingEnv(gym.Env):
    def __init__(self, data_iterator, initial_cash=10000, transaction_cost=0.001):
        super().__init__()
        self.data_iterator = data_iterator
        self.initial_cash = initial_cash
        self.transaction_cost = transaction_cost
        self.num_stocks = data_iterator.num_stocks
        self.lookback = data_iterator.lookback
        self.num_features = data_iterator.num_features

        # Action space for continuous portfolio allocation
        self.action_space = spaces.Box(
            low=0, high=1, shape=(self.num_stocks + 1,), dtype=np.float32
        )

        # Observation space: stock features over the lookback period
        self.observation_space = spaces.Box(
            low=-np.inf, high=np.inf,
            shape=(self.num_stocks, self.lookback, self.num_features), dtype=np.float32
        )

        self.reset()

    def reset(self, seed=None, options=None):
        """Reset the environment at the beginning of an episode"""
        self.data_iterator.reset()
        self.cash = self.initial_cash
        self.holdings = np.zeros(self.num_stocks)

        try:
            self.current_step = next(self.data_iterator)
            return self._get_observation(), {}
        except StopIteration:
            return np.zeros((self.num_stocks, self.lookback, self.num_features), dtype=np.float32), {}

    def _get_observation(self):
        return self.current_step['features'].astype(np.float32)

    def step(self, action):
        """Execute a trade based on action provided by DDPG model"""
        action = np.clip(action, 0, 1)
        action_sum = action.sum()
        if action_sum > 0:
            action /= action_sum

        current_prices = self.current_step['prices']
        portfolio_value = self.cash + np.sum(self.holdings * current_prices)

        target_values = action[1:] * portfolio_value
        current_values = self.holdings * current_prices
        delta_values = target_values - current_values

        transaction_costs = np.abs(delta_values).sum() * self.transaction_cost

        for i in range(self.num_stocks):
            if current_prices[i] > 0:
                self.holdings[i] += delta_values[i] / current_prices[i]

        self.cash = portfolio_value - np.sum(self.holdings * current_prices) - transaction_costs
        old_portfolio_value = portfolio_value

        try:
            self.current_step = next(self.data_iterator)
            new_prices = self.current_step['prices']
            done = False
        except StopIteration:
            new_prices = current_prices
            done = True

        new_portfolio_value = self.cash + np.sum(self.holdings * new_prices)
        reward = (new_portfolio_value / old_portfolio_value) - 1 - (transaction_costs / old_portfolio_value)

        return (
            self._get_observation() if not done else np.zeros_like(self.observation_space.sample()),
            float(reward),
            done,
            False,
            {"portfolio_value": new_portfolio_value}
        )

# model evaluation

Model Evaluation Function for Portfolio Trading
Description:
The evaluate_model function is designed to assess the performance of a trained Deep Reinforcement Learning (DRL) model (such as DDPG) in a simulated stock trading environment. It evaluates how well the model manages a stock portfolio over a validation dataset, calculating key performance metrics like total rewards, portfolio growth, and annualized returns.

This function is essential for validating the profitability and robustness of the trained AI trading model before deploying it in live market conditions.

Key Features:
Validation Dataset:
Uses recent stock data (past 5 years) for evaluation.
Selects top liquid stocks based on trading volume.
Performance Tracking:
Tracks total rewards (cumulative profit/loss).
Logs final portfolio value at the end of each episode.
Computes annualized returns based on trading performance.
Multiple Episodes for Stability:
Evaluates model performance over multiple episodes (default: 10).
Helps measure consistency and reliability of the strategy.
Performance Metrics:
Average total reward across episodes.
Average final portfolio value (growth from initial cash).
Success rate (percentage of episodes with positive returns).
How It Works:
Initialize Evaluation Environment:

Loads validation stock data.
Creates a StockPortfolioIterator for dataset processing.
Sets up the PortfolioTradingEnv to simulate trading.
Run Model Across Multiple Episodes:

Resets the environment at the start of each episode.
Executes trades using the trained model's predicted actions.
Tracks portfolio growth and rewards.
Calculate Performance Metrics:

Computes final portfolio value for each episode.
Derives annualized return using trading duration.
Computes average performance statistics over multiple episodes.
Print Evaluation Summary:

Displays per-episode performance.
Summarizes overall average reward, portfolio value, and success rate.
Ideal Use Case:
This function is ideal for quantitative finance researchers, algorithmic traders, and hedge funds who want to evaluate an AI-powered portfolio optimization model before deploying it in real-world trading.

In [6]:
def evaluate_model(model, data_path, max_stocks=100, lookback=30, episodes=10):
    """Evaluate the model on a validation dataset"""
    print(f"Evaluating model performance...")

    eval_iter = StockPortfolioIterator(
        data_path,
        lookback_window=lookback,
        min_history=100,
        max_stocks=max_stocks,
        train_years=5  # Use more recent data for evaluation
    )

    eval_env = PortfolioTradingEnv(eval_iter)

    total_rewards = []
    portfolio_values = []

    for ep in range(episodes):
        obs, _ = eval_env.reset()
        done = False
        episode_reward = 0
        initial_value = eval_env.initial_cash
        step_count = 0

        while not done:
            action, _ = model.predict(obs, deterministic=True)
            obs, reward, done, _, info = eval_env.step(action)
            episode_reward += reward
            step_count += 1

            if done:
                final_value = info["portfolio_value"]
                portfolio_values.append(final_value)

                # Calculate annualized return
                if step_count > 0:
                    days = step_count
                    annualized_return = ((final_value / initial_value) ** (252 / days) - 1) * 100
                else:
                    annualized_return = 0

                print(f"Episode {ep+1}: Return = {episode_reward:.4f}, "
                      f"Steps = {step_count}, "
                      f"Final Value = ${final_value:.2f}, "
                      f"Annualized Return = {annualized_return:.2f}%")

        total_rewards.append(episode_reward)

    # Calculate summary statistics
    avg_reward = np.mean(total_rewards)
    avg_final_value = np.mean(portfolio_values)
    success_rate = np.sum(np.array(total_rewards) > 0) / len(total_rewards) * 100

    print(f"\nEvaluation Results (over {episodes} episodes):")
    print(f"Average Total Reward: {avg_reward:.4f}")
    print(f"Average Final Portfolio Value: ${avg_final_value:.2f}")
    print(f"Success Rate (positive return): {success_rate:.2f}%")

    return total_rewards, portfolio_values